# 03 — Ajuste de hiperparâmetros (grid search)

> **Apêndice / próximos passos.** O `02_cnn.ipynb` treina *uma* configuração fixa e
> já é um trabalho completo. Este notebook mostra como **escolher hiperparâmetros de
> forma honesta**, usando o conjunto de validação — dando a ele o papel que a
> metodologia descreve: *treino ajusta os pesos, validação ajusta hiperparâmetros,
> teste mede uma única vez, no fim*.

**Como funciona:**
- Um **grid pequeno** (poucas combinações) é treinado e comparado **na validação**.
- Cada tentativa roda com **poucas épocas** (rápido, inclusive na CPU).
- A configuração vencedora é **retreinada** por mais épocas e avaliada **uma única
  vez no teste**.

> ⚠️ **Não rode isto ao vivo numa apresentação.** Na CPU são vários minutos. Rode
> antes, deixe o `results/grid_results.json` salvo e apresente a **tabela** — o valor
> está no raciocínio, não em ver treinando.

In [ ]:
# ── Setup — funciona igual no seu PC e no Google Colab ───────────────────────
# Ajuste REPO_URL após publicar o projeto no GitHub (ver README).
import sys, os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/4rth-g/fashion-mnist-fundamentos-ia.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # No Colab não há o repositório: clonamos para ter o código de src/ e as pastas.
    if not Path("fashion-mnist-fundamentos-ia").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("fashion-mnist-fundamentos-ia")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn"], check=True)

# Torna o pacote src/ importável (local: rodando de notebooks/; Colab: da raiz).
_root = Path.cwd()
_src = _root / "src" if (_root / "src").exists() else _root.parent / "src"
sys.path.insert(0, str(_src))
print("src/ em:", _src)

In [ ]:
import itertools
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from utils import seed_everything, get_device, make_dataloaders, CLASS_NAMES, RESULTS_DIR

seed_everything()
device = get_device()

# Reusa o mesmo split e a mesma augmentation do 02 — muda só o hiperparâmetro.
train_loader, val_loader, test_loader = make_dataloaders(batch_size=128, device=device)
print(f"Batches — treino: {len(train_loader)} | val: {len(val_loader)} | teste: {len(test_loader)}")

## A CNN (parametrizável) e o loop de treino

A arquitetura é a **mesma do `02`**, mas com o `dropout` como parâmetro para o grid
poder variá-lo. `train_and_validate` treina uma configuração e devolve a **melhor
acurácia de validação** — nenhuma referência ao teste aqui.

In [ ]:
class CNN(nn.Module):
    """Mesma CNN do 02_cnn, com dropout parametrizável para o grid search."""

    def __init__(self, dropout: float = 0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


def run_epoch(model, loader, criterion, optimizer=None):
    """Roda uma época. Com `optimizer`, treina; senão, apenas avalia."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            n += images.size(0)
    return total_loss / n, correct / n


def train_and_validate(lr, dropout, weight_decay, epochs, seed=42):
    """Treina uma configuração e devolve (melhor_val_acc, melhores_pesos).

    Mesma semente em todas as configs -> mesma inicialização e mesma ordem de dados,
    para uma comparação justa (a diferença vem só do hiperparâmetro).
    """
    seed_everything(seed)
    model = CNN(dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val, best_state = 0.0, None
    for _ in range(epochs):
        run_epoch(model, train_loader, criterion, optimizer)
        _, val_acc = run_epoch(model, val_loader, criterion)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    return best_val, best_state

## O grid

Enxuto de propósito — o custo é **multiplicativo** (nº de combinações × épocas). Comece
pequeno; só aumente se tiver GPU e tempo.

In [ ]:
# Grid pequeno e CPU-friendly. Descomente valores para ampliar com parcimônia.
LRS = [1e-3, 5e-4]
DROPOUTS = [0.3, 0.5]
WEIGHT_DECAYS = [0.0]        # ex.: adicione 1e-4 para exercitar regularização L2
EPOCHS_SEARCH = 5           # poucas épocas por tentativa; o vencedor retreina mais

grid = list(itertools.product(LRS, DROPOUTS, WEIGHT_DECAYS))
print(f"{len(grid)} combinações × {EPOCHS_SEARCH} épocas cada")

In [ ]:
results = []
best = {"val_acc": 0.0, "state": None}

for lr, dropout, wd in grid:
    val_acc, state = train_and_validate(lr, dropout, wd, EPOCHS_SEARCH)
    results.append({"lr": lr, "dropout": dropout, "weight_decay": wd, "val_acc": val_acc})
    print(f"lr={lr:.0e} | dropout={dropout} | wd={wd} -> val_acc {val_acc:.4f}")
    if val_acc > best["val_acc"]:
        best = {"lr": lr, "dropout": dropout, "weight_decay": wd, "val_acc": val_acc, "state": state}

df = pd.DataFrame(results).sort_values("val_acc", ascending=False).reset_index(drop=True)
print("\nRanking pela validação:")
df

## Retreino do vencedor e avaliação no TESTE (uma única vez)

Só agora, com a configuração escolhida **pela validação**, retreinamos por mais épocas
e tocamos no conjunto de teste — uma vez, para uma estimativa honesta.

In [ ]:
from sklearn.metrics import classification_report, f1_score

print(f"Melhor config (pela validação): lr={best['lr']:.0e}, "
      f"dropout={best['dropout']}, weight_decay={best['weight_decay']} "
      f"(val_acc {best['val_acc']:.4f})")

EPOCHS_FINAL = 10
final_val, final_state = train_and_validate(
    best["lr"], best["dropout"], best["weight_decay"], EPOCHS_FINAL)

model = CNN(dropout=best["dropout"]).to(device)
model.load_state_dict(final_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        all_preds.append(model(images.to(device)).argmax(1).cpu())
        all_labels.append(labels)
all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

test_acc = (all_preds == all_labels).mean()
macro_f1 = f1_score(all_labels, all_preds, average="macro")
print(f"\nTESTE — acurácia {test_acc:.4f} | F1 macro {macro_f1:.4f}\n")
print(classification_report(all_labels, all_preds, digits=4, target_names=CLASS_NAMES))

In [ ]:
# Persiste o resultado do grid para auditoria e para apresentar sem rodar ao vivo.
out = {
    "grid": results,
    "best_config": {k: best[k] for k in ("lr", "dropout", "weight_decay")},
    "best_val_accuracy": float(final_val),
    "test_accuracy": float(test_acc),
    "macro_f1": float(macro_f1),
    "epochs_search": EPOCHS_SEARCH,
    "epochs_final": EPOCHS_FINAL,
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
}
with open(RESULTS_DIR / "grid_results.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print(f"Resultados salvos em {RESULTS_DIR / 'grid_results.json'}")

## Conclusão

- O grid search **usa a validação** para escolher hiperparâmetros; o teste continua
  intocado até a avaliação final — sem vazamento.
- O ganho costuma ser **modesto** (afinar `lr`/`dropout` move a acurácia ~0,5–1,5%);
  o valor principal é **metodológico**, não o número.
- Para a apresentação: mostre a **tabela** (`df`) e o `results/grid_results.json`
  já computados; não rode o grid ao vivo.